In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# data setup

this notebook loads both datasets, explores their structure,
builds a bridge between tabular and image data, and saves
the final matched dataset.

datasets used:
- vehicle sales csv (558k rows) — tabular car sales data
- stanford cars dataset (16k images) — car images with class labels

output:
- df_matched.csv (40,658 rows) — rows that have both price data and a matching car image

In [2]:
import os

BASE = '/content/drive/MyDrive/CarPricePrediction/'

# Check tabular
csv_path = BASE + 'data/tabular/car_prices.csv.zip'
zip_path = BASE + 'data/images/archive.zip'

print("CSV exists:", os.path.exists(csv_path))
print("ZIP exists:", os.path.exists(zip_path))

# Check sizes
print("CSV size:", os.path.getsize(csv_path) / 1e6, "MB")
print("ZIP size:", os.path.getsize(zip_path) / 1e6, "MB")

CSV exists: True
ZIP exists: True
CSV size: 19.753181 MB
ZIP size: 1959.400431 MB


## extract stanford cars images to local disk

we extract to /content/ (local disk) not drive because local disk
is 10x faster for reading images during training.
note: local disk resets every session — re-run this cell each time.

In [3]:
import zipfile

LOCAL_IMG = '/content/stanford_cars/'
os.makedirs(LOCAL_IMG, exist_ok=True)

print("Extracting to local disk...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(LOCAL_IMG)

print("Done!")
print("Files:", len(os.listdir(LOCAL_IMG)))

Extracting to local disk...
Done!
Files: 3


In [10]:
import pandas as pd

df = pd.read_csv(csv_path)
print("Tabular data shape:", df.shape)
print("Columns:", df.columns.tolist())

Tabular data shape: (558837, 16)
Columns: ['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate']


In [11]:
print("Unique makes:", df['make'].nunique())
print("Unique models:", df['model'].nunique())
print("Unique years:", sorted(df['year'].unique()))
print("\nTop 10 makes:")
print(df['make'].value_counts().head(10))

Unique makes: 96
Unique models: 973
Unique years: [np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]

Top 10 makes:
make
Ford         93554
Chevrolet    60197
Nissan       53946
Toyota       39871
Dodge        30710
Honda        27206
Hyundai      21816
BMW          20719
Kia          18077
Chrysler     17276
Name: count, dtype: int64


## load stanford car class names

stanford has 196 car classes. each class = one specific car
(make + model + body style + year).
we load these to see what's on the image side of our bridge.

In [12]:
import scipy.io as sio

devkit_path = os.path.join(LOCAL_IMG, 'car_devkit', 'devkit')
meta = sio.loadmat(os.path.join(devkit_path, 'cars_meta.mat'))
class_names = meta['class_names'][0]

print("Total Stanford classes:", len(class_names))
print("\nFirst 10 classes:")
for i in range(10):
    print(f"  {i+1}: {class_names[i][0]}")

Total Stanford classes: 196

First 10 classes:
  1: AM General Hummer SUV 2000
  2: Acura RL Sedan 2012
  3: Acura TL Sedan 2012
  4: Acura TL Type-S 2008
  5: Acura TSX Sedan 2012
  6: Acura Integra Type R 2001
  7: Acura ZDX Hatchback 2012
  8: Aston Martin V8 Vantage Convertible 2012
  9: Aston Martin V8 Vantage Coupe 2012
  10: Aston Martin Virage Convertible 2012


## build the bridge

strategy: match csv rows to stanford classes using make + year + model.
a csv row gets an image only if stanford has that exact make/model/year combination.

In [13]:
# Build stanford lookup: (make, year) -> class_name
stanford_classes = []
for i in range(len(class_names)):
    name = class_names[i][0]
    words = name.split()
    make = words[0]
    year = int(words[-1])
    stanford_classes.append({
        'class_name': name,
        'make': make,
        'year': year
    })

stanford_df = pd.DataFrame(stanford_classes)
print("Stanford classes as dataframe:")
print(stanford_df.head(10))

Stanford classes as dataframe:
                                 class_name   make  year
0                AM General Hummer SUV 2000     AM  2000
1                       Acura RL Sedan 2012  Acura  2012
2                       Acura TL Sedan 2012  Acura  2012
3                      Acura TL Type-S 2008  Acura  2008
4                      Acura TSX Sedan 2012  Acura  2012
5                 Acura Integra Type R 2001  Acura  2001
6                  Acura ZDX Hatchback 2012  Acura  2012
7  Aston Martin V8 Vantage Convertible 2012  Aston  2012
8        Aston Martin V8 Vantage Coupe 2012  Aston  2012
9      Aston Martin Virage Convertible 2012  Aston  2012


In [14]:
print(df['make'].value_counts()[df['make'].value_counts() > 100])

make
Ford             93554
Chevrolet        60197
Nissan           53946
Toyota           39871
Dodge            30710
Honda            27206
Hyundai          21816
BMW              20719
Kia              18077
Chrysler         17276
Mercedes-Benz    17141
Jeep             15372
Infiniti         15305
Volkswagen       12581
Lexus            11861
GMC              10613
Mazda             8362
Cadillac          7519
Acura             5901
Audi              5869
Lincoln           5757
Buick             5107
Subaru            5043
Ram               4574
Pontiac           4497
Mitsubishi        4140
Volvo             3788
MINI              3224
Saturn            2841
Mercury           1992
Land Rover        1735
Scion             1687
Jaguar            1420
Porsche           1383
Suzuki            1073
FIAT               865
HUMMER             805
Saab               484
ford               443
smart              396
chevrolet          390
Oldsmobile         364
dodge              245
chrysl

In [15]:
# Standardize make to title case
df['make'] = df['make'].str.strip().str.title()
df['model'] = df['model'].str.strip().str.title()

# Verify fix
print(df['make'].value_counts().head(10))
print("\nTotal unique makes after fix:", df['make'].nunique())

make
Ford         93997
Chevrolet    60587
Nissan       54017
Toyota       39966
Dodge        30955
Honda        27351
Hyundai      21836
Bmw          20793
Kia          18084
Chrysler     17485
Name: count, dtype: int64

Total unique makes after fix: 66


In [16]:
# Standardize stanford makes too
for s in stanford_classes:
    s['make'] = s['make'].strip().title()

stanford_df = pd.DataFrame(stanford_classes)

# Find matching makes between CSV and Stanford
csv_makes = set(df['make'].unique())
stanford_makes = set(stanford_df['make'].unique())

overlap_makes = csv_makes.intersection(stanford_makes)
print("Makes in CSV:", len(csv_makes))
print("Makes in Stanford:", len(stanford_makes))
print("Overlapping makes:", len(overlap_makes))
print("\nOverlap:", sorted(overlap_makes))

Makes in CSV: 67
Makes in Stanford: 49
Overlapping makes: 41

Overlap: ['Acura', 'Audi', 'Bentley', 'Bmw', 'Buick', 'Cadillac', 'Chevrolet', 'Chrysler', 'Daewoo', 'Dodge', 'Ferrari', 'Fiat', 'Fisker', 'Ford', 'Geo', 'Gmc', 'Honda', 'Hummer', 'Hyundai', 'Infiniti', 'Isuzu', 'Jaguar', 'Jeep', 'Lamborghini', 'Lincoln', 'Mazda', 'Mercedes-Benz', 'Mini', 'Mitsubishi', 'Nissan', 'Plymouth', 'Porsche', 'Ram', 'Rolls-Royce', 'Scion', 'Smart', 'Suzuki', 'Tesla', 'Toyota', 'Volkswagen', 'Volvo']


In [17]:
# Filter CSV to only overlapping makes
df_filtered = df[df['make'].isin(overlap_makes)].copy()
print("CSV rows before filter:", len(df))
print("CSV rows after filter:", len(df_filtered))
print("Rows dropped:", len(df) - len(df_filtered))

CSV rows before filter: 558837
CSV rows after filter: 500943
Rows dropped: 57894


In [18]:
# Get overlapping years
csv_years = set(df_filtered['year'].unique())
stanford_years = set(stanford_df['year'].unique())

overlap_years = csv_years.intersection(stanford_years)
print("Years in CSV:", sorted(csv_years))
print("Years in Stanford:", sorted(stanford_years))
print("Overlapping years:", sorted(overlap_years))

Years in CSV: [np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]
Years in Stanford: [np.int64(1991), np.int64(1993), np.int64(1994), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012)]
Overlapping years: [np.int64(1991), np.int64(1993), np.int64(1994), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), n

In [19]:
# Filter by overlapping years
df_filtered = df_filtered[df_filtered['year'].isin(overlap_years)].copy()
print("Rows after year filter:", len(df_filtered))
print("\nYear distribution:")
print(df_filtered['year'].value_counts().sort_index())

Rows after year filter: 283094

Year distribution:
year
1991       60
1993      154
1994      314
1997     1183
1998     1657
1999     2516
2000     4024
2001     5677
2002     8477
2006    22699
2007    26353
2008    27448
2009    18517
2010    24063
2011    44830
2012    95122
Name: count, dtype: int64


In [20]:
# create a lookup dictionary
# key = (make, year), value = list of matching stanford classes
stanford_lookup = {}

for _, row in stanford_df.iterrows():
    key = (row['make'], row['year'])
    if key not in stanford_lookup:
        stanford_lookup[key] = []
    stanford_lookup[key].append(row['class_name'])

# test it with a few examples
print(stanford_lookup.get(('Ford', 2012), 'not found'))
print(stanford_lookup.get(('Toyota', 2012), 'not found'))
print(stanford_lookup.get(('Bmw', 2012), 'not found'))

[np.str_('Ford F-450 Super Duty Crew Cab 2012'), np.str_('Ford Edge SUV 2012'), np.str_('Ford F-150 Regular Cab 2012'), np.str_('Ford E-Series Wagon Van 2012'), np.str_('Ford Fiesta Sedan 2012')]
[np.str_('Toyota Sequoia SUV 2012'), np.str_('Toyota Camry Sedan 2012'), np.str_('Toyota Corolla Sedan 2012'), np.str_('Toyota 4Runner SUV 2012')]
[np.str_('BMW ActiveHybrid 5 Sedan 2012'), np.str_('BMW 1 Series Convertible 2012'), np.str_('BMW 1 Series Coupe 2012'), np.str_('BMW 3 Series Sedan 2012'), np.str_('BMW 3 Series Wagon 2012'), np.str_('BMW X6 SUV 2012'), np.str_('BMW M3 Coupe 2012'), np.str_('BMW X3 SUV 2012'), np.str_('BMW Z4 Convertible 2012')]


In [21]:
# match csv row to stanford class using make + year + model
def find_stanford_class(make, model, year):
    # get all stanford classes for this make + year
    candidates = stanford_lookup.get((make, int(year)), [])

    if not candidates:
        return None

    # check if model name appears in any candidate
    model_lower = str(model).lower()
    for candidate in candidates:
        if model_lower in candidate.lower():
            return candidate

    # no model match found
    return None

# test it
print(find_stanford_class('Ford', 'Edge', 2012))
print(find_stanford_class('Toyota', 'Camry', 2012))
print(find_stanford_class('Bmw', '3 Series', 2012))
print(find_stanford_class('Ford', 'Mustang', 2012))

Ford Edge SUV 2012
Toyota Camry Sedan 2012
BMW 3 Series Sedan 2012
None


In [22]:
print(stanford_lookup.get(('Ford', 2012), 'not found'))

[np.str_('Ford F-450 Super Duty Crew Cab 2012'), np.str_('Ford Edge SUV 2012'), np.str_('Ford F-150 Regular Cab 2012'), np.str_('Ford E-Series Wagon Van 2012'), np.str_('Ford Fiesta Sedan 2012')]


In [23]:
# apply matching function to entire filtered dataframe
print("matching csv rows to stanford classes...")
print("this will take a minute...")

df_filtered['stanford_class'] = df_filtered.apply(
    lambda row: find_stanford_class(row['make'], row['model'], row['year']),
    axis=1
)

# see how many rows got a match
matched = df_filtered['stanford_class'].notna().sum()
unmatched = df_filtered['stanford_class'].isna().sum()

print(f"matched: {matched}")
print(f"unmatched: {unmatched}")
print(f"match rate: {matched/len(df_filtered)*100:.1f}%")

matching csv rows to stanford classes...
this will take a minute...
matched: 40658
unmatched: 242436
match rate: 14.4%


In [24]:
# look at unmatched rows - what makes/models are they
unmatched_df = df_filtered[df_filtered['stanford_class'].isna()]
print("top unmatched models:")
print(unmatched_df.groupby(['make', 'model']).size().sort_values(ascending=False).head(20))

top unmatched models:
make        model         
Nissan      Altima            10980
Ford        Fusion             6205
            F-150              6028
Honda       Civic              5465
Ford        Escape             5241
Bmw         3 Series           4962
Nissan      Maxima             4803
Infiniti    G Sedan            4437
Nissan      Rogue              4342
Toyota      Camry              4034
Chevrolet   Impala             3943
Ford        Focus              3839
Chevrolet   Malibu             3708
Honda       Accord             3698
Dodge       Grand Caravan      3199
Bmw         5 Series           3141
Nissan      Sentra             3092
Volkswagen  Jetta              3013
Chevrolet   Silverado 1500     2956
Ford        Explorer           2844
dtype: int64


In [25]:
# check what stanford has for nissan
print(stanford_lookup.get(('Nissan', 2012), 'not found'))

[np.str_('Nissan Leaf Hatchback 2012'), np.str_('Nissan NV Passenger Van 2012'), np.str_('Nissan Juke Hatchback 2012')]


In [26]:
sample = df_filtered[df_filtered['model'] == '3 Series'].head(3)
print(sample[['make', 'model', 'year', 'stanford_class']])

    make     model  year           stanford_class
756  Bmw  3 Series  2012  BMW 3 Series Sedan 2012
757  Bmw  3 Series  2012  BMW 3 Series Sedan 2012
765  Bmw  3 Series  2012  BMW 3 Series Sedan 2012


In [27]:
# keep only rows that have a stanford match
df_matched = df_filtered[df_filtered['stanford_class'].notna()].copy()
print("final multimodal dataset size:", len(df_matched))
print("\nmake distribution:")
print(df_matched['make'].value_counts().head(10))
print("\nyear distribution:")
print(df_matched['year'].value_counts().sort_index())

final multimodal dataset size: 40658

make distribution:
make
Ford             5666
Chevrolet        4663
Honda            4551
Hyundai          4335
Toyota           4060
Mercedes-Benz    2368
Jeep             2273
Bmw              2116
Dodge            1958
Acura            1531
Name: count, dtype: int64

year distribution:
year
1993        4
2001        2
2007     4567
2008      382
2009      356
2010     1593
2011      866
2012    32888
Name: count, dtype: int64


## save matched dataset

keeping only rows with a stanford match.
these 40,658 rows have both price data and a car image —
this is what we feed into our multimodal model.

match rate of 14.4% is expected — stanford only covers 196 specific car classes
out of thousands in our csv.

In [28]:
# save matched dataset to drive
save_path = BASE + 'data/tabular/df_matched.csv'
df_matched.to_csv(save_path, index=False)
print("saved matched dataset to drive")
print("shape:", df_matched.shape)
print("columns:", df_matched.columns.tolist())

saved matched dataset to drive
shape: (40658, 17)
columns: ['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate', 'stanford_class']
